# V2 Audio Patch

Patches `train_features_v2.pt` and `test_features_v2.pt` that were generated with all-zero audio.

**What this does (and does NOT do):**
- **Does NOT** re-run CLIP or ImageBind vision — those are already correct in the stored files.
- **Does** re-run VGGish audio extraction → overwrites `z_aud` [N, 128]
- **Does** re-run ImageBind audio extraction → fuses with the stored `v_teacher` (= raw IB_vision) to produce the proper multimodal teacher: `normalize((IB_vision + IB_audio) / 2)`
- **Does** update `has_audio` [N, bool]

**Key insight:** When `has_audio=False` the stored `v_teacher[i]` equals the raw unnormalized `IB_vision[i]` (no normalization was applied in `build_teacher`). So it can be reused directly — no need to re-run the vision encoder.

Checkpoints every 200 videos. Safe to restart if the kernel dies.

## Step 1: Install Dependencies

In [ ]:
!pip install -q git+https://github.com/facebookresearch/ImageBind.git
!pip install -q resampy soundfile tqdm
# VGGish via torch.hub (no separate install needed)

## Step 2: Imports & GPU Check

In [ ]:
import os
import json
import subprocess
import zipfile
import urllib.request

import torch
import torch.nn.functional as F
from tqdm import tqdm

from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType
import imagebind.data as ib_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Use a local temp dir — avoids /tmp permission issues on some Kaggle environments
TMP_DIR = "./tmp_audio"
os.makedirs(TMP_DIR, exist_ok=True)

## Step 3: Download MSR-VTT Videos (if not already present)

In [ ]:
os.makedirs("msrvtt", exist_ok=True)

urls = {
    "msrvtt/msrvtt_train_7k.json": "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/msrvtt_train_7k.json",
    "msrvtt/msrvtt_test_1k.json":  "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/msrvtt_test_1k.json",
    "msrvtt/MSRVTT_Videos.zip":    "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/MSRVTT_Videos.zip",
}

for path, url in urls.items():
    if not os.path.exists(path):
        print(f"Downloading {path}...")
        urllib.request.urlretrieve(url, path)
        print("Done.")
    else:
        print(f"Already exists: {path}")

video_dir = "msrvtt/video"
if not os.path.exists(video_dir):
    print("Extracting videos...")
    with zipfile.ZipFile("msrvtt/MSRVTT_Videos.zip", "r") as zf:
        zf.extractall("msrvtt")
    print("Done.")
else:
    n = len(os.listdir(video_dir))
    print(f"Video dir exists with {n} files.")

## Step 3b: Diagnose Audio Availability

Run this on a handful of videos before the full patch to confirm audio streams exist.
If all come back `has_audio=False`, the video files don't have audio — no point patching.

In [ ]:
def probe_audio(video_path):
    """Returns (has_stream, wav_size_bytes) by actually trying to extract a wav."""
    # Method 1: ffprobe stream check
    cmd = [
        "ffprobe", "-v", "error",
        "-select_streams", "a:0",
        "-show_entries", "stream=codec_name",
        "-of", "default=noprint_wrappers=1",
        video_path,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    has_stream = bool(result.stdout.strip())

    # Method 2: actually extract and check size
    tmp_wav = os.path.join(TMP_DIR, "probe_test.wav")
    cmd2 = f'ffmpeg -y -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{tmp_wav}"'
    res2 = subprocess.run(cmd2, shell=True, capture_output=True, text=True)
    wav_size = os.path.getsize(tmp_wav) if os.path.exists(tmp_wav) else 0
    if os.path.exists(tmp_wav):
        os.remove(tmp_wav)

    return has_stream, wav_size, res2.returncode


sample_vids = sorted(os.listdir("msrvtt/video"))[:10]
print(f"{'Video':<30} {'ffprobe_has_stream':>18} {'wav_bytes':>10} {'ffmpeg_rc':>10}")
print("-" * 72)
for v in sample_vids:
    has_s, wav_sz, rc = probe_audio(os.path.join("msrvtt/video", v))
    print(f"{v:<30} {str(has_s):>18} {wav_sz:>10} {rc:>10}")

## Step 4: Load Models

Only VGGish and ImageBind needed — CLIP is skipped (vision features not re-extracted).

In [ ]:
# VGGish
vggish = torch.hub.load("harritaylor/torchvggish", "vggish", trust_repo=True)
vggish.eval().to(device)
print("VGGish loaded.")

# ImageBind
ib_model = imagebind_model.imagebind_huge(pretrained=True)
ib_model.eval().to(device)
print("ImageBind loaded.")

## Step 5: Audio Extraction Helpers

In [ ]:
def extract_vggish_audio(video_path, tmp_name="vggish_audio.wav"):
    """Returns [128] VGGish embedding, or zeros if audio unavailable."""
    tmp = os.path.join(TMP_DIR, tmp_name)
    if os.path.exists(tmp):
        os.remove(tmp)
    cmd = f'ffmpeg -y -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{tmp}"'
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if not os.path.exists(tmp) or os.path.getsize(tmp) < 1000:
        if os.path.exists(tmp):
            os.remove(tmp)
        return torch.zeros(128)
    try:
        with torch.no_grad():
            feat = vggish.forward(tmp)
            if feat.ndim > 1:
                feat = feat.mean(dim=0)
        os.remove(tmp)
        return feat.cpu()
    except Exception as e:
        if os.path.exists(tmp):
            os.remove(tmp)
        return torch.zeros(128)


def extract_imagebind_audio(video_path, tmp_name="ib_audio.wav"):
    """Returns [1024] ImageBind audio embedding, or None if extraction fails."""
    tmp = os.path.join(TMP_DIR, tmp_name)
    if os.path.exists(tmp):
        os.remove(tmp)
    cmd = f'ffmpeg -y -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{tmp}"'
    subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if not os.path.exists(tmp) or os.path.getsize(tmp) < 1000:
        if os.path.exists(tmp):
            os.remove(tmp)
        return None
    try:
        audio_data = ib_data.load_and_transform_audio_data([tmp], device)
        inputs = {ModalityType.AUDIO: audio_data}
        with torch.no_grad():
            emb = ib_model(inputs)[ModalityType.AUDIO]
        os.remove(tmp)
        return emb.squeeze(0).cpu()  # [1024]
    except Exception as e:
        print(f"    IB audio error for {os.path.basename(video_path)}: {e}")
        if os.path.exists(tmp):
            os.remove(tmp)
        return None


def build_teacher(v_vis, v_aud):
    """Fuse IB vision + audio into normalized multimodal teacher."""
    if v_aud is None:
        return v_vis  # unchanged — raw IB_vision, no normalization
    combined = (v_vis + v_aud) / 2.0
    return F.normalize(combined, dim=-1)  # unit-norm multimodal embedding


print("Helpers defined.")

## Step 6: Patch Function

Iterates over all videos, re-extracts audio, and patches `z_aud`, `v_teacher`, `has_audio`.
Checkpoints every 200 videos — resume-safe if kernel dies.

In [ ]:
CHECKPOINT_EVERY = 200


def patch_features(features_pt, split_json, video_dir):
    """
    Loads an existing V2 feature file and patches z_aud, v_teacher, has_audio in-place.
    Saves the result back to the same path (with a .bak backup first).
    """
    ckpt_pt = features_pt.replace(".pt", "_patch_ckpt.pt")

    print(f"\n{'='*60}")
    print(f"Patching: {features_pt}")
    print(f"{'='*60}")

    # Load original features
    orig = torch.load(features_pt, map_location="cpu")
    v_teachers_orig = orig["v_teacher"].to(torch.float32)  # [N, 1024] = raw IB_vision
    video_ids_all   = list(orig["video_ids"])              # ordered list
    N = len(video_ids_all)
    print(f"Loaded {N} samples. Current audio coverage: "
          f"{orig['has_audio'].sum().item()}/{N} (expected 0)")

    # Build video_id -> filename map
    with open(split_json) as f:
        jdata = json.load(f)
    vid_to_file = {item["video_id"]: item["video"] for item in jdata}

    # Resume from checkpoint if it exists
    if os.path.exists(ckpt_pt):
        ckpt = torch.load(ckpt_pt, map_location="cpu")
        patched_z_aud     = list(ckpt["z_aud"])      # list of [128]
        patched_v_teacher = list(ckpt["v_teacher"])  # list of [1024]
        patched_has_audio = list(ckpt["has_audio"])  # list of bool tensors
        n_done = len(patched_z_aud)
        print(f"Resuming from checkpoint: {n_done}/{N} already patched.")
    else:
        patched_z_aud, patched_v_teacher, patched_has_audio = [], [], []
        n_done = 0

    remaining_ids = video_ids_all[n_done:]
    print(f"Videos left to process: {len(remaining_ids)}")

    n_audio_found = sum(h.item() for h in patched_has_audio)
    errors = 0

    for i, vid in enumerate(tqdm(remaining_ids, desc=os.path.basename(features_pt))):
        global_idx = n_done + i
        v_vis = v_teachers_orig[global_idx]  # stored raw IB_vision [1024]

        filename = vid_to_file.get(vid)
        if not filename:
            print(f"\n  Warning: {vid} not in JSON — keeping zero audio.")
            patched_z_aud.append(torch.zeros(128))
            patched_v_teacher.append(v_vis)
            patched_has_audio.append(torch.tensor(False))
            errors += 1
            continue

        video_path = os.path.join(video_dir, filename)
        if not os.path.exists(video_path):
            print(f"\n  Warning: {video_path} missing — keeping zero audio.")
            patched_z_aud.append(torch.zeros(128))
            patched_v_teacher.append(v_vis)
            patched_has_audio.append(torch.tensor(False))
            errors += 1
            continue

        try:
            # VGGish student audio
            z_aud = extract_vggish_audio(video_path)

            # ImageBind audio for teacher fusion
            v_aud = extract_imagebind_audio(video_path)

            # Fuse teacher if audio available
            v_teach = build_teacher(v_vis, v_aud)
            audio_ok = (v_aud is not None)

            patched_z_aud.append(z_aud)
            patched_v_teacher.append(v_teach)
            patched_has_audio.append(torch.tensor(audio_ok, dtype=torch.bool))

            if audio_ok:
                n_audio_found += 1

        except Exception as e:
            print(f"\n  Error on {vid}: {e}")
            patched_z_aud.append(torch.zeros(128))
            patched_v_teacher.append(v_vis)
            patched_has_audio.append(torch.tensor(False))
            errors += 1

        # Checkpoint
        if (i + 1) % CHECKPOINT_EVERY == 0:
            _save_ckpt(ckpt_pt, patched_z_aud, patched_v_teacher, patched_has_audio)
            print(f"  Checkpoint @ {len(patched_z_aud)}/{N} | "
                  f"audio={n_audio_found} | errors={errors}")

    # Assemble final patched dict
    patched = dict(orig)  # carries over z_img, video_ids unchanged
    patched["z_aud"]     = torch.stack(patched_z_aud).to(torch.float32)
    patched["v_teacher"] = torch.stack(patched_v_teacher).to(torch.float32)
    patched["has_audio"] = torch.stack(patched_has_audio)

    # Backup original, then overwrite
    bak = features_pt.replace(".pt", "_orig.bak.pt")
    if not os.path.exists(bak):
        import shutil
        shutil.copy2(features_pt, bak)
        print(f"  Backup saved: {bak}")

    torch.save(patched, features_pt)
    print(f"\nPatched file saved: {features_pt}")
    print(f"  z_aud      : {patched['z_aud'].shape}  "
          f"(non-zero: {(torch.norm(patched['z_aud'], dim=1) > 1e-5).sum().item()})")
    print(f"  v_teacher  : {patched['v_teacher'].shape}")
    print(f"  has_audio  : {patched['has_audio'].sum().item()}/{N} "
          f"({100*patched['has_audio'].float().mean():.1f}%)")
    print(f"  errors     : {errors}")

    # Remove checkpoint on success
    if os.path.exists(ckpt_pt):
        os.remove(ckpt_pt)
        print("  Checkpoint removed.")


def _save_ckpt(path, z_auds, v_teachers, has_audios):
    torch.save({
        "z_aud":     torch.stack(z_auds),
        "v_teacher": torch.stack(v_teachers),
        "has_audio": torch.stack(has_audios),
    }, path)


print("Patch function defined.")

## Step 7: Run Patch

Test set first (~1000 videos, ~30 min on T4), then train (~7010 videos, ~2h on T4).
Only VGGish + IB audio — no vision re-extraction, so roughly half the time of full extraction.

In [ ]:
patch_features(
    features_pt="test_features_v2.pt",
    split_json="msrvtt/msrvtt_test_1k.json",
    video_dir="msrvtt/video",
)

patch_features(
    features_pt="train_features_v2.pt",
    split_json="msrvtt/msrvtt_train_7k.json",
    video_dir="msrvtt/video",
)

## Step 8: Verify & Download

In [ ]:
from IPython.display import FileLink, display

for fname in ["test_features_v2.pt", "train_features_v2.pt"]:
    if not os.path.exists(fname):
        print(f"MISSING: {fname}")
        continue
    d = torch.load(fname, map_location="cpu")
    n_aud = d["has_audio"].sum().item()
    n_tot = len(d["video_ids"])
    non_zero_aud = (torch.norm(d["z_aud"].float(), dim=1) > 1e-5).sum().item()
    print(f"\n{fname}")
    print(f"  z_img      : {d['z_img'].shape}  (dtype: {d['z_img'].dtype})")
    print(f"  z_aud      : {d['z_aud'].shape}  non-zero={non_zero_aud}/{n_tot}")
    print(f"  v_teacher  : {d['v_teacher'].shape}")
    print(f"  has_audio  : {n_aud}/{n_tot} ({100*n_aud/n_tot:.1f}%)")
    display(FileLink(fname, result_html_prefix=f"Download {fname}: "))

# Also list any backup files
print("\nFiles in working dir:")
print([f for f in os.listdir(".") if f.endswith(".pt") or f.endswith(".bak.pt")])